# 01 — Acquire OSM Source Data

## Research purpose

Acquire and prepare the OpenStreetMap data-centre records and nearby building footprints used in the analysis.


## Imports

Load the libraries used for acquisition and geometry normalization.


In [1]:
import json
import time
from pathlib import Path
from typing import Any

import geopandas as gpd
import pandas as pd
import requests
from pyproj import Geod
from shapely.geometry import GeometryCollection, LineString, MultiPolygon, Point, Polygon, mapping
from shapely.ops import unary_union
from tqdm.auto import tqdm

/opt/homebrew/Caskroom/miniforge/base/envs/hyperscale-db-test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Project location and paths

Identify the project directory and define the paths for the OSM snapshots, normalised records and review tables.


In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    
if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
OSM_RAW_DIR = DATA_RAW / "osm"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

raw_path = OSM_RAW_DIR / "osm_datacenters_overpass_raw.json"
context_path = OSM_RAW_DIR / "osm_building_footprints_raw.geojson"
normalized_csv = DATA_INTERIM / "osm_datacenters_normalized.csv"
normalized_geojson = DATA_INTERIM / "osm_datacenters_geometry_normalized.geojson"
unusable_path = DATA_INTERIM / "osm_datacenters_unusable_review.csv"
failure_path = DATA_INTERIM / "osm_context_building_fetch_failures.csv"

## Create project folders

Create the folders used to store source data, intermediate tables, model results and figures. Existing files are left in place.


In [3]:
project_folders = [
    "data/raw/osm",
    "data/raw/overture",
    "data/raw/maps",
    "data/raw/renewables",
    "data/raw/demand",
    "data/manual",
    "data/interim",
    "data/processed",
    "data/final/candidate_hours_6h",
    "data/final/candidate_hours_9h",
    "visualizations/flexibility/common",
    "visualizations/flexibility/greedy",
    "visualizations/flexibility/optimised",
    "visualizations/flexibility/comparison",
    "visualizations/projections/greedy",
    "visualizations/projections/optimised",
    "visualizations/projections/comparison",
]

for folder in project_folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

## Acquisition settings

Set the geometry reference, context search distances and Overpass query for data-centre features. Use the saved snapshots by default and allow live requests when a snapshot is missing or `FORCE_REFRESH` is enabled.


In [ ]:
# Geometry and context coverage.
GEOD = Geod(ellps="WGS84")
NODE_CONTEXT_RADIUS_M = 150
BBOX_PADDING_DEGREES = 0.0005

# Acquisition controls: prefer the supplied snapshots.
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
USER_AGENT = "master-thesis pipeline"
FETCH_CONTEXT_BUILDINGS = True
CONTEXT_FETCH_SLEEP_SECONDS = 1.0
FORCE_REFRESH = False

DATA_CENTER_QUERY = """
[out:json][timeout:900];
(
  nwr["telecom"="data_center"];
  nwr["telecom"="data_centre"];
  nwr["building"="data_center"];
  nwr["building"="data_centre"];
  nwr["man_made"="data_center"];
  nwr["man_made"="data_centre"];
);
out tags center geom;
""".strip()

## Geometry normalisation

Define functions to convert OSM elements into geometry, preserve their tags and distinguish building footprints from site boundaries. Polygon areas are measured on the WGS84 ellipsoid, and representative points provide coordinates where needed. For relations, use closed outer ways without reconstructing inner rings.


In [5]:
# Serialize OSM tags consistently while preserving their original values.
def tags_to_json(tags: dict[str, Any] | None) -> str:
    return json.dumps(tags or {}, ensure_ascii=False, sort_keys=True)


# Convert supported Overpass elements to geometry, using closed outer ways for relations.
def geometry_from_overpass_element(element: dict[str, Any]):
    """Return a Shapely geometry from an Overpass JSON element when possible."""
    element_type = element.get("type")
    if element_type == "node":
        return Point(float(element["lon"]), float(element["lat"]))
    geom = element.get("geometry") or []
    coords = [(float(p["lon"]), float(p["lat"])) for p in geom if "lon" in p and "lat" in p]
    if element_type == "way" and len(coords) >= 2:
        if len(coords) >= 4 and coords[0] == coords[-1]:
            return Polygon(coords)
        return LineString(coords)
    if element_type == "relation":
        polygons = []
        for member in element.get("members", []):
            if member.get("type") != "way" or member.get("role") not in {"outer", ""}:
                continue
            member_geom = member.get("geometry") or []
            member_coords = [
                (float(p["lon"]), float(p["lat"]))
                for p in member_geom
                if "lon" in p and "lat" in p
            ]
            if len(member_coords) >= 4 and member_coords[0] == member_coords[-1]:
                polygons.append(Polygon(member_coords))
        if polygons:
            merged = unary_union(polygons)
            if isinstance(merged, (Polygon, MultiPolygon)):
                return merged
            return GeometryCollection(polygons)
    return None


# Measure polygon area on the WGS84 ellipsoid and leave non-polygon area missing.
def area_m2(geometry) -> float | None:
    """Calculate geodesic polygon area in square metres."""
    if geometry is None or geometry.is_empty:
        return None
    if geometry.geom_type == "Polygon":
        area, _ = GEOD.geometry_area_perimeter(geometry)
        return abs(float(area))
    if geometry.geom_type == "MultiPolygon":
        return sum(area_m2(part) or 0.0 for part in geometry.geoms)
    return None


# Return a representative-point coordinate when the source lacks a direct centre.
def centroid_lat_lon(geometry) -> tuple[float | None, float | None]:
    if geometry is None or geometry.is_empty:
        return None, None
    point = geometry.representative_point()
    return float(point.y), float(point.x)


# Classify geometry as a point, building footprint, site boundary, or other geometry.
def geometry_kind(tags: dict[str, Any], geometry) -> str:
    if geometry is None:
        return "missing"
    if geometry.geom_type == "Point":
        return "point"
    if tags.get("building") and tags.get("building") != "no":
        return "building_footprint"
    if tags.get("landuse"):
        return "site_or_landuse_polygon"
    if geometry.geom_type in {"Polygon", "MultiPolygon"}:
        return "unclassified_polygon"
    return geometry.geom_type.lower()

## Load or acquire OSM records

Load the saved Overpass response when it is available. If the snapshot is missing or `FORCE_REFRESH` is enabled, request the data-centre records and save the response.


In [6]:
def fetch_overpass(query: str, timeout: int = 900) -> dict:
    response = requests.post(
        OVERPASS_URL,
        data={"data": query},
        headers={"User-Agent": USER_AGENT},
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json()

if raw_path.exists() and not FORCE_REFRESH:
    raw = json.loads(raw_path.read_text(encoding="utf-8"))
    acquisition_mode = "submitted_snapshot"
else:
    raw = fetch_overpass(DATA_CENTER_QUERY)
    if raw_path.exists() and not FORCE_REFRESH:
        raise FileExistsError("Refusing to overwrite the frozen OSM snapshot")
    raw_path.write_text(json.dumps(raw, ensure_ascii=False), encoding="utf-8")
    acquisition_mode = "live_overpass"
print(f"Acquisition mode: {acquisition_mode}; elements: {len(raw.get('elements', [])):,}")

Acquisition mode: submitted_snapshot; elements: 4,348


## Normalise source records

Convert usable OSM elements into a geographic table with their source identifiers, tags, coordinates and polygon areas. Keep elements without usable geometry in a separate review table and stop if no usable records remain.


In [7]:
def parse_osm_elements(elements, context_source_id=None):
    records = []
    geometries = []
    for element in elements:
        tags = element.get("tags", {}) or {}
        geometry = geometry_from_overpass_element(element)
        if geometry is None:
            continue
        lat, lon = centroid_lat_lon(geometry)
        center = element.get("center") or {}
        lat = center.get("lat", element.get("lat", lat))
        lon = center.get("lon", element.get("lon", lon))
        element_type = element.get("type")
        osm_id = element.get("id")
        osm_ref = f"{element_type}/{osm_id}"
        source_id = f"osm_{element_type}_{osm_id}"
        record = {
            "source": "osm",
            "source_id": source_id,
            "osm_ref": osm_ref,
            "osm_id": osm_id,
            "osm_type": element_type,
            "name": tags.get("name"),
            "operator": tags.get("operator"),
            "owner": tags.get("owner"),
            "brand": tags.get("brand"),
            "website": tags.get("website"),
            "latitude": lat,
            "longitude": lon,
            "building": tags.get("building"),
            "landuse": tags.get("landuse"),
            "telecom": tags.get("telecom"),
            "man_made": tags.get("man_made"),
            "building_levels": tags.get("building:levels"),
            "height": tags.get("height"),
            "matched_tag": ";".join(f"{k}={v}" for k, v in tags.items() if k in {"telecom", "building", "man_made"}),
            "osm_tags": tags_to_json(tags),
            "geometry_kind": geometry_kind(tags, geometry),
            "polygon_area_m2": area_m2(geometry),
            "geometry_geojson": json.dumps(mapping(geometry), ensure_ascii=False),
            "context_source_id": context_source_id,
        }
        records.append(record)
        geometries.append(geometry)
    return gpd.GeoDataFrame(records, geometry=geometries, crs="EPSG:4326")

osm_gdf = parse_osm_elements(raw.get("elements", []))
usable_refs = set(osm_gdf["osm_ref"].astype(str)) if not osm_gdf.empty else set()
unusable = pd.DataFrame([
    {
        "osm_type": element.get("type"),
        "osm_id": element.get("id"),
        "osm_ref": f"{element.get('type')}/{element.get('id')}",
        "reason": "no_usable_geometry",
    }
    for element in raw.get("elements", [])
    if f"{element.get('type')}/{element.get('id')}" not in usable_refs
], columns=["osm_type", "osm_id", "osm_ref", "reason"])
print(f"Normalized records: {len(osm_gdf):,}; unusable elements: {len(unusable):,}")
if osm_gdf.empty:
    unusable.to_csv(unusable_path, index=False)
    raise ValueError(f"No usable OSM data-centre geometries were found; see {unusable_path}")

Normalized records: 4,286; unusable elements: 62


## Save normalised records

Save the normalised attributes, geographic features and unusable-record review. Display counts by geometry and OSM element type to inspect the available evidence.


In [8]:
csv_columns = [column for column in osm_gdf.columns if column != "geometry"]
osm_gdf[csv_columns].to_csv(normalized_csv, index=False)
osm_gdf.to_file(normalized_geojson, driver="GeoJSON")
unusable.to_csv(unusable_path, index=False)

print(osm_gdf["geometry_kind"].value_counts(dropna=False).to_string())
print(osm_gdf["osm_type"].value_counts(dropna=False).to_string())

geometry_kind
building_footprint         3150
point                       834
site_or_landuse_polygon     172
unclassified_polygon        130
osm_type
way     3452
node     834


## Building footprint queries

Define the queries used to find nearby building footprints. Point records use a 150 m search radius, while other supported geometries use a padded bounding box.


In [9]:
def context_query_for_row(row):
    geom = row.geometry
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == "Point":
        return f'''[out:json][timeout:180];
way["building"](around:{NODE_CONTEXT_RADIUS_M},{row.latitude},{row.longitude});
out tags center geom;'''
    minx, miny, maxx, maxy = geom.bounds
    west, south = minx - BBOX_PADDING_DEGREES, miny - BBOX_PADDING_DEGREES
    east, north = maxx + BBOX_PADDING_DEGREES, maxy + BBOX_PADDING_DEGREES
    return f'''[out:json][timeout:180];
way["building"]({south},{west},{north},{east});
out tags center geom;'''

## Load or acquire building footprints

Reuse the saved building footprints unless `FORCE_REFRESH` is enabled. When acquisition is needed, request footprints around sites and points and retain their links to the original OSM records. Record failed requests and stop before saving if a request fails or no usable footprints are returned.


In [10]:
if context_path.exists() and not FORCE_REFRESH:
    buildings_gdf = gpd.read_file(context_path)
    print(f"Using submitted context snapshot: {len(buildings_gdf):,} buildings")
elif FETCH_CONTEXT_BUILDINGS:
    targets = osm_gdf[
        osm_gdf["geometry_kind"].isin(["site_or_landuse_polygon", "unclassified_polygon", "point"])
        & osm_gdf["source_id"].notna()
    ].copy()
    parts, failures = [], []
    for _, row in tqdm(targets.iterrows(), total=len(targets)):
        query = context_query_for_row(row)
        if not query:
            continue
        try:
            response = fetch_overpass(query, timeout=240)
            part = parse_osm_elements(response.get("elements", []), context_source_id=row["source_id"])
            if not part.empty:
                parts.append(part)
        except Exception as exc:
            failures.append({"source_id": row["source_id"], "error": str(exc)})
        time.sleep(CONTEXT_FETCH_SLEEP_SECONDS)
    if failures:
        pd.DataFrame(failures).to_csv(failure_path, index=False)
        raise RuntimeError(f"{len(failures)} context queries failed; see {failure_path}")
    if not parts:
        raise ValueError(
            f"No usable context building footprints were returned for {len(targets):,} source records; "
            "the context snapshot was not written."
        )
    buildings_gdf = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry="geometry", crs="EPSG:4326")
    buildings_gdf.to_file(context_path, driver="GeoJSON")
    print(f"Saved {len(buildings_gdf):,} context buildings")
else:
    raise FileNotFoundError(
        "Context footprint acquisition is required because the snapshot is missing or FORCE_REFRESH=True, "
        "but FETCH_CONTEXT_BUILDINGS=False."
    )

Using submitted context snapshot: 34,810 buildings
